# PathFinder - Internship Resume Matching & Shortlisting Model
This notebook demonstrates the end-to-end data pipeline, feature engineering, cross-validation, and performance evaluation for PathFinder's candidate shortlisting model.

### Objective:
Predict the probability of a candidate being shortlisted for an internship based on their resume content, required skill overlap, semantic fit, project portfolio, and overall completeness.

## 1. Imports and Environment Setup

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import joblib

print('Environment initialized successfully.')

## 2. Load Dataset
The dataset contains 1,200 curated resume-job pairs spanning five tech domains: Software Engineering, Data Science, UI/UX, DevOps, and QA.

In [2]:
data_path = os.path.join('..', 'data', 'resume_internship_dataset.csv')
df = pd.read_csv(data_path)
print(f'Total records: {len(df)}')
df.head()

## 3. Exploratory Data Analysis & Domain Distribution

In [3]:
print('Domain Distribution:')
print(df['domain'].value_counts())

print('\nTarget Distribution (Shortlisted):')
print(df['shortlisted'].value_counts(normalize=True))

print('\nFeature Summary Statistics:')
df[['semantic_similarity', 'skill_overlap_ratio', 'missing_core_skills', 'project_count', 'cv_completeness']].describe()

## 4. Train-Test Split (80/20 Stratified)

In [4]:
features = [
    'semantic_similarity',
    'skill_overlap_ratio',
    'missing_core_skills',
    'project_count',
    'cv_completeness'
]
target = 'shortlisted'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training samples: {len(X_train)}, Testing samples: {len(X_test)}')

## 5. Model Training & 5-Fold Cross Validation
We train a `RandomForestClassifier` with balanced depth to avoid overfitting and validate generalization using 5-fold cross-validation.

In [5]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')

print(f'5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

# Fit on full training set
model.fit(X_train, y_train)
print('Model training complete.')

## 6. Test Set Evaluation & Metrics

In [6]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not Shortlisted', 'Shortlisted']))
print(f'Test Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Test Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Test Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'Test F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'Test ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}')

## 7. Feature Importance Analysis
Analyzing which features contribute most to the shortlisting decision.

In [7]:
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance_df)

# Plot feature importances
plt.figure(figsize=(8, 4))
plt.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1], color='#3b82f6')
plt.title('PathFinder Shortlisting Model - Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Export Model & Artifacts

In [8]:
model_save_path = os.path.join('..', 'models', 'shortlist_model.pkl')
joblib.dump(model, model_save_path)
print(f'Model artifact persisted to {model_save_path}')